# Scratchpad
A place to experiment with agent calls in a modular way before inserting them into the codebase.

In [ ]:
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
import os

load_dotenv(override=True)

In [ ]:
def get_default_model(use_local_llm: bool=os.getenv("USE_LOCAL_LLM")):
    if use_local_llm:
        return ChatOllama(model="gemma4", temperature=0)
    else:
        return "openai:gpt-5.6-luna"


In [ ]:
def run_test_prompt(agent):
    result = agent.invoke({"messages": [{"role": "user", "content": "What is the Model Context Protocol, in two sentences?"}]})
    print(result["messages"][-1].content)

system_prompt="You are a helpful assistant who answers concisely."

# print("Running test prompt on remote LLM...")
# run_test_prompt(
#     create_agent(
#         system_prompt=system_prompt,
#         model=get_default_model(use_local_llm=False),
#     )
# )

# print("Running test prompt on local LLM...")
# run_test_prompt(
#     create_agent(
#         system_prompt=system_prompt,
#         model=get_default_model(use_local_llm=True),
#     )
# )


In [ ]:
import pymupdf

def load_pdf(filename: str) -> str:
    doc = pymupdf.open(filename)
    pages = [page.get_text() for page in doc]
    return "\n\n".join(pages)

In [ ]:
# print(load_pdf("sandbox/press_release.pdf"))

In [ ]:
import pdfplumber

def load_pdf_with_markdown_tables(pdf_path: str) -> list[str]:
    full_content = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            full_content.append(f"--- Page {page_num} ---")
            
            # Extract tables on the page
            tables = page.extract_tables()
            
            if tables:
                for table in tables:
                    # Clean out None values and formatting quirks
                    clean_table = [[str(cell or "").strip().replace("\n", " ") for cell in row] for row in table]
                    
                    if not clean_table or not clean_table[0]:
                        continue
                        
                    # Format as Markdown Table
                    header = clean_table[0]
                    markdown_table = "| " + " | ".join(header) + " |\n"
                    markdown_table += "| " + " | ".join(["---"] * len(header)) + " |\n"
                    
                    for row in clean_table[1:]:
                        markdown_table += "| " + " | ".join(row) + " |\n"
                    
                    full_content.append(markdown_table)
            
            # Extract non-table text
            page_text = page.extract_text()
            if page_text:
                full_content.append(page_text)
    return full_content

def merge_pages(pages: list[str]) -> str:
    return "\n\n".join(pages)

In [ ]:
# print(merge_pages(load_pdf_with_markdown_tables("sandbox/press_release.pdf")))

In [ ]:
import asyncio
from quarterly_report_parse_result import QuarterlyReportParseResult

parser_system_prompt = """
You are a financial analyst.

You are given a 10Q document.

You are tasked with extracting financial statements from 10Q documents, so that your team can run fundamental analysis using that data.

Documents often have multiple values for each field, which corresponds to different dates. 
You should only extract the value that is for the most recent date.

Sometimes reports will use a dash or a line to represent zero. 
Just use 0. Do not try to infer the value.

Sometimes values will be surrounded by parentheses. This represents a negative value. 
Use the negative value of the number inside the parentheses.
"""
parser_agent = create_agent(
    system_prompt=parser_system_prompt,
    model=get_default_model(),
    response_format=QuarterlyReportParseResult,
)

# Runs the parser on one page of the document at a time
# At the end, it merges the results together
async def run_parser_page_by_page() -> QuarterlyReportParseResult:
    quarterly_report_pages = load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf")
    requests = [run_parser_on_page_of_quarterly_report(page) for page in quarterly_report_pages]
    results = await asyncio.gather(*requests)

    merged_result = QuarterlyReportParseResult()
    for result in results:
        merged_result = merged_result.merge(result)
    return merged_result

async def run_parser_on_page_of_quarterly_report(page: str):
    message = f"""
Here is a page from a 10Q document that I would like you to parse for financial statements:

{page}
"""

    result = await parser_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

# Runs the parser on the whole document at once
def run_parser_on_whole_document() -> QuarterlyReportParseResult:
    message = f"""
Here is a 10Q document that I would like you to parse:

{merge_pages(load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf"))}
"""

    result = parser_agent.invoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]




In [ ]:
from dataclasses import dataclass

class QuarterInfo(BaseModel):
    month: str = Field(description="The last month of the reported quarter. Example: 'March'")
    end_date: str = Field(description="The last day of the reported quarter. Example: 'March 31, 2026'")


@dataclass
class QuarterParsingParameters:
    quarter: str
    end_date: str
    columns_to_use: list[str]

def get_quarter_parsing_parameters(quarter_info: QuarterInfo) -> QuarterParsingParameters:
    match quarter_info.month.lower():
        case "march" | "april" | "may":
            return QuarterParsingParameters(quarter="Q1", end_date=quarter_info.end_date, columns_to_use=["13-weeks", "3 months", "year to date", "ytd"])
        case "june" | "july" | "august":
            return QuarterParsingParameters(quarter="Q2", end_date=quarter_info.end_date, columns_to_use=["26-weeks", "6 months", "year to date", "ytd"])
        case "september" | "october" | "november":
            return QuarterParsingParameters(quarter="Q3", end_date=quarter_info.end_date, columns_to_use=["39-weeks", "9 months", "year to date", "ytd"])
        case "december" | "january" | "february":
            return QuarterParsingParameters(quarter="Q4", end_date=quarter_info.end_date, columns_to_use=["52-weeks", "12 months", "full year", "annual", "for the year ended", "year to date", "ytd"])
        case _:
            raise ValueError(f"Invalid month: {quarter_info.month}") 
            

In [ ]:
class FinancialReportTable(BaseModel):
    description: str = Field(description="A description of the information contained in the table. Example: 'Consolidated Statements of Cash Flows'")
    markdown_table: str = Field(description="The markdown table that contains the information. Example: '| Cash Flow | Amount |'")
    flagged_as_irrelevant: bool = Field(description="Whether the table is irrelevant to our analysis. Should be false by default.", default=False)

class FinancialReportTables(BaseModel):
    tables: list[FinancialReportTable] = Field(description="A list of tables from a 10Q document. Example: '[FinancialReportTable(description='Consolidated Statements of Cash Flows', markdown_table='| Cash Flow | Amount |'), FinancialReportTable(description='Consolidated Balance Sheet', markdown_table='| Cash Flow | Amount |')]'")

def turn_table_to_string(table: FinancialReportTable) -> str:
    return f"""
Here is a description of the table:
{table.description}

Here is the table:
{table.markdown_table}
"""

def turn_tables_to_string(tables: list[FinancialReportTable]) -> str:
    return "\n===============\n".join([turn_table_to_string(table) for table in tables])


In [ ]:
from table_parsing import (
    replace_dash_only_table_cells_with_zero_values,
    replace_parenthesized_numbers_with_negative_values,
)

    
data_extraction_system_prompt = """
You are a financial analyst.

You are given a 10Q document.

10Q documents are lengthy documents that are sometimes difficult for agents to process.
Your job is to trim down the document to the most relevant parts and clean it up, so that it is easier for the parsing agent to process.
"""
table_extraction_agent = create_agent(
    system_prompt=data_extraction_system_prompt,
    model=get_default_model(),
    response_format=FinancialReportTables,
)

table_cleaning_agent = create_agent(
    system_prompt=data_extraction_system_prompt,
    model=get_default_model(),
    response_format=FinancialReportTable,
)

quarter_info_extraction_agent = create_agent(
    system_prompt=data_extraction_system_prompt,
    model=get_default_model(),
    response_format=QuarterInfo,
)

async def extract_quarter_info(pages: list[str]) -> QuarterInfo:
    for page in pages:
        quarter_info = await extract_quarter_info_from_page(page)
        if quarter_info.end_date is not None and quarter_info.month is not None:
            # The end date should usually be on the first page of the document.
            # The loop is to handle unusual documents that have the end date on a different page.
            return quarter_info
    raise ValueError("No quarter info found in the document")

async def extract_quarter_info_from_page(page: str) -> QuarterInfo:
    message = f"""
I have a page from a 10Q report. I'm trying to figure out the last day that this report covers.

There should be a statement in the document that says something like "For the quarter ended [end date]".
If you see that statement on this page, please extract that end date. Do not try to infer the end date.

Here is the page:
{page}
"""

    result = await quarter_info_extraction_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

async def extract_tables_from_report_page(page: str, quarter_info: QuarterParsingParameters) -> list[FinancialReportTable]:
    message = f"""
I have a page from a 10Q document. I would like you to extract any financial tables that you see, so that we can run fundamental analysis on them.

The quarter that we are analyzing ends on:
{quarter_info.end_date}
If a table is about a different data, please flag it as irrelevant.

The time period that we care about is:
{quarter_info.columns_to_use}
If a table is not about that time period, please flag it as irrelevant.

========================================

Here is the page:
{page}
"""

    result = await table_extraction_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"].tables


async def clean_table(table: FinancialReportTable, quarter_info: QuarterParsingParameters) -> FinancialReportTable:
    message = f"""
I have a table from a 10Q document. I want you to remove information that is irrelevant to our analysis. Below are the instructions.

Sometimes a table will contain extra columns for other dates. 
Here is the date that we care about:
{quarter_info.end_date}

Remove any columns that are not for the date that we care about.

Sometimes a table will contain extra columns for different time periods.
Here are the time periods that we care about:
{quarter_info.columns_to_use}

Remove any columns that are not for the time periods that we care about.

========================================

Here is the table that I want you to refine:

{table.markdown_table}
"""

    trimmed_table = (await table_cleaning_agent.ainvoke({"messages": [{"role": "user", "content": message}]}))["structured_response"]

    cleaned_markdown_table = replace_parenthesized_numbers_with_negative_values(trimmed_table.markdown_table)
    cleaned_markdown_table = replace_dash_only_table_cells_with_zero_values(cleaned_markdown_table)

    return trimmed_table.model_copy(update={"markdown_table": cleaned_markdown_table})

async def extract_tables_from_report(pages: list[str], quarter_info: QuarterParsingParameters) -> list[FinancialReportTable]:
    tables = []
    results = await asyncio.gather(*[extract_cleaned_tables_from_page(page, quarter_info) for page in pages])
    for result in results:
        tables.extend(result)
    return tables

async def extract_cleaned_tables_from_page(page: str, quarter_info: QuarterParsingParameters) -> list[FinancialReportTable]:
    tables_from_page = await extract_tables_from_report_page(page, quarter_info)
    tables_from_page = [table for table in tables_from_page if not table.flagged_as_irrelevant]
    cleaned_tables = await asyncio.gather(*[clean_table(table, quarter_info) for table in tables_from_page])
    return cleaned_tables

async def run_parser_table_by_table() -> QuarterlyReportParseResult:
    pages = load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf")
    quarter_info = get_quarter_parsing_parameters(await extract_quarter_info(pages))
    tables = await extract_tables_from_report(pages, quarter_info)

    merged_result = QuarterlyReportParseResult()

    # Run the parser on each table one-by-one
    results = await asyncio.gather(*[run_parser_on_table(table) for table in tables])

    # Merge the parse results together
    for result in results:
        merged_result = merged_result.merge(result)

    return merged_result


async def run_parser_on_table(table: FinancialReportTable) -> QuarterlyReportParseResult:
    message = f"""
Here is a table from a 10Q document. I would like you to parse it for financial statements:

{turn_table_to_string(table)}
"""

    result = await parser_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

async def run_parser_on_all_tables_at_once() -> QuarterlyReportParseResult:
    pages = load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf")
    quarter_info = get_quarter_parsing_parameters(await extract_quarter_info(pages))
    tables = await extract_tables_from_report(pages, quarter_info)

    message = f"""
Here is are all of the tables from a 10Q document. I would like you to parse them for financial statements:

{turn_tables_to_string(tables)}
"""

    result = await parser_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

In [ ]:
# Run comparisons
from quarterly_report_parse_result import count_populated_fields, get_diffs


quarterly_report_result = run_parser_on_whole_document()
table_by_table_quarterly_report_result = await run_parser_table_by_table()
all_tables_at_once_quarterly_report_result = await run_parser_on_all_tables_at_once()

print(f"Quarterly Report Result: {count_populated_fields(quarterly_report_result)}")
print(f"Table by Table Quarterly Report Result: {count_populated_fields(table_by_table_quarterly_report_result)}")
print(f"All Tables at Once Quarterly Report Result: {count_populated_fields(all_tables_at_once_quarterly_report_result)}")

diffs = get_diffs(quarterly_report_result, table_by_table_quarterly_report_result)
print(f"\n\n{len(diffs)} differences found between running the parser on the whole document and running it on each table")
print(f"Diffs: {"\n".join(diffs)}")

diffs = get_diffs(quarterly_report_result, all_tables_at_once_quarterly_report_result)
print(f"\n\n{len(diffs)} differences found between running the parser on the whole document and running it on all of the tables at once")
print(f"Diffs: {"\n".join(diffs)}")

diffs = get_diffs(table_by_table_quarterly_report_result, all_tables_at_once_quarterly_report_result)
print(f"\n\n{len(diffs)} differences found between running the parser on each table and running it on all of the tables at once")
print(f"Diffs: {"\n".join(diffs)}")


Notes
* cost_of_goods_sold seems to be inconsistently parsed (i think that the parsing instructions need to be updated)

Prompt:
The next field I'd like to review is weighted_average_shares. It can either be basic or diluted. I'd like you to:

  1. Identify how that value is being used in run_all_calculations.py
  2. Determine whether we should grab the basic value, diluted value, or both for those calculations